# Word-Level Audio Perception MVP on OCI Data Science

This notebook builds a transparent Gradio demonstrator for recording or uploading a spoken sentence, transcribing it with word-level timestamps, adjusting experimental frequency settings, generating transformed audio, and collecting the listener's response.

Run the cells in order in an OCI Data Science Notebook Session with outbound internet access. Use only non-sensitive audio that you are authorized to process, particularly when `demo.launch(share=True)` creates a temporary external Gradio link. This notebook is an educational MVP, not a validated scientific, clinical, hearing-assessment, or production system.

In [ ]:
!pip install gradio

In [ ]:
!pip install openai-whisper
!pip install gtts librosa soundfile

In [ ]:
!pip install faster-whisper
!pip install ffmpeg-python

> **FFmpeg environment note —** FFmpeg installation can vary depending on the OCI Data Science image, Conda environment, operating-system architecture, and available shared libraries. This MVP retains the static installation method that worked in the environment in which it was originally developed and tested. Treat it as a tested reference rather than a universal installation procedure. Before continuing, verify that `ffmpeg -version` runs successfully. If it does not, install a build compatible with your selected Notebook Session environment and architecture. Troubleshooting every possible FFmpeg configuration is outside the scope of this MVP tutorial.

In [ ]:
!wget https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-i686-static.tar.xz
!tar -xf ffmpeg-release-i686-static.tar.xz
!cp ffmpeg*/ffmpeg /usr/local/bin/
!chmod +x /usr/local/bin/ffmpeg

In [ ]:
!ffmpeg -version

In [ ]:
import huggingface_hub
print(huggingface_hub.__version__)

In [ ]:
import os
import gradio as gr
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
import re

# Whisper
try:
    import whisper
    _HAS_WHISPER = True
    _WHISPER_MODEL = whisper.load_model("small")
except Exception:
    _HAS_WHISPER = False
    _WHISPER_MODEL = None

# Audio filters
def _butter_bandpass_sos(low_hz, high_hz, sr, order=4):
    nyq = sr * 0.5
    low = max(1.0, low_hz) / nyq
    high = min(high_hz, sr * 0.49) / nyq
    low = min(max(low, 1e-4), 0.99)
    high = min(max(high, low + 1e-4), 0.9999)
    return butter(order, [low, high], btype="band", output="sos")

def _bandpass_filter(y, sr, fc_hz, bw_hz):
    if bw_hz <= 0:
        return y.copy()
    low = fc_hz - bw_hz / 2.0
    high = fc_hz + bw_hz / 2.0
    sos = _butter_bandpass_sos(low, high, sr, order=4)
    return sosfiltfilt(sos, y).astype(np.float32)

def _shift_to_freq(seg, sr, target_freq_hz):
    if target_freq_hz <= 1:
        return seg
    try:
        f0_est = librosa.yin(seg, fmin=80, fmax=8000, sr=sr)
        f_orig = np.median(f0_est[np.isfinite(f0_est)])
    except Exception:
        f_orig = 1000.0
    if not np.isfinite(f_orig) or f_orig <= 0:
        f_orig = 1000.0
    ratio = target_freq_hz / f_orig
    n_steps = 12.0 * np.log2(ratio)
    n_steps = np.clip(n_steps, -24.0, 24.0)
    seg_shift = librosa.effects.pitch_shift(y=seg, sr=sr, n_steps=n_steps)
    return seg_shift.astype(np.float32)

def transcribe_file(audio_path: str):
    if audio_path is None or not os.path.exists(audio_path):
        raise gr.Error("Aucun fichier audio valide.")
    if not _HAS_WHISPER or _WHISPER_MODEL is None:
        raise gr.Error("Whisper indisponible.")
    res = _WHISPER_MODEL.transcribe(audio_path, language="fr", word_timestamps=True, fp16=False)
    full_text = (res.get("text") or "").strip()
    words = []
    for seg in res.get("segments", []):
        for w in seg.get("words", []):
            token = (w.get("word") or "").strip()
            if token:
                words.append({"word": token, "start": float(w["start"]), "end": float(w["end"])})
    if len(words) == 0:
        tokens = [t for t in full_text.split() if t]
        duration = librosa.get_duration(path=audio_path)
        edges = np.linspace(0.0, duration, len(tokens) + 1)
        for i, tok in enumerate(tokens):
            words.append({"word": tok, "start": float(edges[i]), "end": float(edges[i+1])})
    rows = [[w["word"], 1000, 500, w["start"], w["end"]] for w in words]
    df = pd.DataFrame(rows, columns=["Mot", "Fréquence (Hz)", "Bande (Hz)", "Start", "End"])
    return df, full_text

def _safe_float(val, default=1000.0):
    try:
        if val in [None, "", np.nan]:
            return float(default)
        return float(val)
    except Exception:
        return float(default)

def clean_dataframe(df):
    df["Fréquence (Hz)"] = df["Fréquence (Hz)"].apply(lambda x: _safe_float(x, 1000))
    df["Bande (Hz)"] = df["Bande (Hz)"].apply(lambda x: _safe_float(x, 500))
    df["Start"] = df["Start"].apply(lambda x: _safe_float(x, 0))
    df["End"] = df["End"].apply(lambda x: _safe_float(x, 0))
    return df

def generate_full_audio(audio_path: str, df_clean):
    y, sr = librosa.load(audio_path, sr=None, mono=True)
    out = []
    for _, row in df_clean.iterrows():
        f0 = float(row["Fréquence (Hz)"])
        bw = float(row["Bande (Hz)"])
        t0 = float(row["Start"])
        t1 = float(row["End"])
        i0 = max(0, int(round(t0 * sr)))
        i1 = min(len(y), int(round(t1 * sr)))
        if i1 <= i0:
            continue
        seg = y[i0:i1].astype(np.float32)
        seg_shift = _shift_to_freq(seg, sr, target_freq_hz=f0)
        seg_colored = _bandpass_filter(seg_shift, sr, fc_hz=f0, bw_hz=bw)
        out.append(seg_colored)
    y_out = np.concatenate(out).astype(np.float32) if out else np.zeros(1, dtype=np.float32)
    return sr, y_out

def _verify_transcription(user_text, ref_text):
    def clean(text):
        text = text.lower()
        text = re.sub(r"[^\w\s]", "", text)
        return text.split()
    user_words = clean(user_text)
    ref_words = clean(ref_text)
    if not ref_words:
        return "⚠️ Texte de référence vide.", ""
    correct = sum(1 for w in user_words if w in ref_words)
    score = correct / len(ref_words)
    percent = round(score * 100, 2)
    if score >= 0.9:
        result = f"✅ Réussite ! Score : {percent}%"
    else:
        result = f"❌ Échec. Score : {percent}%"
    return result, ""

with gr.Blocks(title="Word-Level Audio Perception MVP") as demo:
    state_audio_path = gr.State(None)
    state_concat = gr.State(None)
    state_target_text = gr.State("")
    state_table_validated = gr.State(None)

    with gr.Tabs():
        with gr.Tab("Administrateur"):
            audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath",
                                label="Uploader un fichier audio ou enregistrer")

            table_out = gr.Dataframe(
                headers=["Mot", "Fréquence (Hz)", "Bande (Hz)", "Start", "End"],
                interactive=True,
                type="array",
                label="Transcription (éditer les valeurs au besoin)"
            )

            btn_sync = gr.Button("✅ Valider les modifications du tableau")
            pitch_plot = gr.Plot(label="📈 Fréquence fondamentale estimée (original)")
            pitch_plot_concat = gr.Plot(label="📈 Fréquence fondamentale estimée (concaténé)")

            with gr.Row():
                btn_trans = gr.Button("Transcrire", variant="primary")
                btn_generate = gr.Button("Générer audio concaténé", variant="secondary")

            audio_out = gr.Audio(label="🎧 Audio filtré concaténé (résultat)", type="numpy")
            btn_send = gr.Button("➡️ Envoyer à l’utilisateur", variant="primary")

        with gr.Tab("Utilisateur"):
            user_audio = gr.Audio(label="🎧 Audio à écouter", type="numpy", interactive=False)
            user_input = gr.Textbox(label="Tapez la phrase entendue", lines=2)
            btn_verify = gr.Button("Vérifier")
            user_result = gr.Markdown()
            with gr.Accordion("Référence (admin)", open=False):
                ref_text = gr.Textbox(label="Texte attendu", interactive=False)

    def _on_transcribe(path):
        df, phrase = transcribe_file(path)
        y, sr = librosa.load(path, sr=None)
        f0 = librosa.yin(y, fmin=80, fmax=8000, sr=sr)
        times = librosa.times_like(f0, sr=sr)
        f0_clean = np.where(np.isfinite(f0), f0, np.nan)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(times, f0_clean, label="Fréquence fondamentale (Hz)", color="purple")
        ax.set_xlabel("Temps (s)")
        ax.set_ylabel("Fréquence (Hz)")
        ax.set_title("Contour de fréquence fondamentale (original)")
        ax.grid(True)
        ax.legend()
        return df.values.tolist(), path, phrase, fig

    btn_trans.click(
        fn=_on_transcribe,
        inputs=[audio_in],
        outputs=[table_out, state_audio_path, state_target_text, pitch_plot],
    )

    def _sync_table(df):
        df_copy = [list(row) for row in df]
        return df_copy, df_copy

    btn_sync.click(
        fn=_sync_table,
        inputs=[table_out],
        outputs=[state_table_validated, table_out],
    )

    def _on_generate(path, df_raw):
        df_clean = pd.DataFrame(df_raw, columns=["Mot", "Fréquence (Hz)", "Bande (Hz)", "Start", "End"])
        df_clean = clean_dataframe(df_clean)
        sr, y_out = generate_full_audio(path, df_clean)

        f0 = librosa.yin(y_out, fmin=80, fmax=8000, sr=sr)
        times = librosa.times_like(f0, sr=sr)
        f0_clean = np.where(np.isfinite(f0), f0, np.nan)

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(times, f0_clean, label="Fréquence fondamentale (Hz)", color="darkgreen")
        ax.set_xlabel("Temps (s)")
        ax.set_ylabel("Fréquence (Hz)")
        ax.set_title("Contour de fréquence fondamentale (concaténé)")
        ax.grid(True)
        ax.legend()

        return (sr, y_out), (sr, y_out), fig

    btn_generate.click(
        fn=_on_generate,
        inputs=[state_audio_path, state_table_validated],
        outputs=[audio_out, state_concat, pitch_plot_concat],
    )

    def _send_to_user(audio_concat, text):
        if audio_concat is None:
            raise gr.Error("Générez d’abord l’audio concaténé.")
        return audio_concat, text

    btn_send.click(
        fn=_send_to_user,
        inputs=[state_concat, state_target_text],
        outputs=[user_audio, ref_text],
    )

    btn_verify.click(
        fn=_verify_transcription,
        inputs=[user_input, ref_text],
        outputs=[user_result, user_input],
    )

demo.launch(share=True)
